## IMPORTING LIBRARIES

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import librosa
import librosa.display
import IPython.display as ipd

## Getting all directory name

In [ ]:
os.listdir('/kaggle/input/competitions/birdclef-2026')

In [ ]:
df_train=pd.read_csv('/kaggle/input/competitions/birdclef-2026/train.csv')

In [ ]:
df_train.head()

In [ ]:
df_train.info()

In [ ]:
folders=['sample_submission.csv',
 'taxonomy.csv',
 'train_audio',
 'train_soundscapes_labels.csv',
 'train_soundscapes',
 'train.csv',
 'recording_location.txt',
 'test_soundscapes']

# Testing on one audio file

In [ ]:
test_audio='/kaggle/input/competitions/birdclef-2026/train_audio/24279/iNat125223.ogg'

# GETTING NUMPY ARRAY IN DATA FOR VISUALIZING THE SIGNALS USING LIBROSA

In [ ]:
plt.figure(figsize=(14,5))
data,sample_rate=librosa.load(test_audio)
librosa.display.waveshow(data,sr=sample_rate)
ipd.Audio(test_audio)

# Spectrogram - Picture of Sound
## STFT- short time fourier transform
1. takes small chunk
2. finds frequencies inside chunk
3. moves slightly forward
4. repeats
``frame_length=320`` Analyze 320 audio samples at one time
``frame_step=32`` Means take 32 samples and then move forward

## After these steps we would have complex number 
### Because Fourier transform stores:
1. magnitude
2. phase
``tensorflow.abs(spectrogram)`` convert complex numbers into magnitude values like ``3+2j = 3.6`` WE GET THIS BY ``abs(3+2j) = sqrt(3² + 2²)``

## At last we expanded the dimension
### Because CNN require height,width,channel 
BEFORE ``(125,123) - > (125,123,1)``

In [ ]:
spectrogram=tf.signal.stft(data,frame_length=320,
                                 frame_step=32)
spectrogram=tf.abs(spectrogram)
spectrogram=tf.expand_dims(spectrogram,axis=2)

## Picture of Spectrogram

In [ ]:
plt.figure(figsize=(14,5))

plt.imshow(
    tf.transpose(spectrogram)[0],
    aspect='auto',
    origin='lower'
)

plt.colorbar()
plt.show()

# Load the file
### Converting the audio file into signal array

In [18]:
def load_audio(filename):
    data,sample_rate=librosa.load(filename,
                                  sr=16000,
                                  mono=True)
    return data

## Preprocesing
1. It takes audio file convert audio to signal array
2. Then it crop first 5 sec of audio and rest is dumpped
3. If the size of audio was less then 5 sec the remaining space is filled with zeros
4. Converting array into spectrogram

In [20]:
def preprocess(filename,label):
    data,sample_rate=load_audio(filename)
    if len(data)<16000:
        spectrogram=tf.zeros([2491,257,1],dtype=tf.float32)
        return spectrogram
    data=data[:80000]
    pad_length = 80000 - tf.shape(data)[0]  # how many zeros needed
    zero_padding = tf.zeros([pad_length], dtype=tf.float32)
    data=tf.concat([data,zero_padding],0)
    spectrogram=tf.signal.stft(data,frame_length=320,
                                 frame_step=32)
    spectrogram=tf.abs(spectrogram)
    spectrogram=tf.expand_dims(spectrogram,axis=2)

    return spectrogram,label
    

### Checking the audio files, max size, min size and mean size for preprocessing

In [ ]:
!pip install mutagen

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
from mutagen.oggvorbis import OggVorbis

train_audio = '/kaggle/input/competitions/birdclef-2026/train_audio'
TARGET_SR = 16000  # Your target sample rate

# Gather all full file paths first
all_files = []
for folder in os.listdir(train_audio):
    folder_path = os.path.join(train_audio, folder)
    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            all_files.append(os.path.join(folder_path, file))

def get_audio_length(file_path):
    try:
        audio = OggVorbis(file_path)
        # audio.info.length gives duration in seconds
        # Multiply by target sample rate to get the sample length
        return int(audio.info.length * TARGET_SR)
    except Exception as e:
        return 0

# Use multithreading to read headers in parallel
with ThreadPoolExecutor() as executor:
    lengths = list(executor.map(get_audio_length, all_files))

# Filter out any failed reads (0) if necessary
lengths = [l for l in lengths if l > 0]

In [21]:
print(f'MEAN Size of audio : {tf.math.reduce_mean(lengths)/16000} sec')
print(f'MIN Size of audio  : {tf.math.reduce_min(lengths)/16000} sec')
print(f'MAX Size of audio  : {tf.math.reduce_max(lengths)/16000} sec')

MEAN Size of audio : 34.8825625 sec
MIN Size of audio  : 0.008 sec
MAX Size of audio  : 6881.097 sec
